# 29a — Autoencoders

Corresponds to: Note 29 — Autoencoders

**Libraries introduced in this notebook:** No new libraries — PyTorch (`torch`, `torch.nn`) was introduced in Note 09.

**What this notebook demonstrates:**
1. Forward pass trace matching the Note 29 worked example (encoder → latent code → decoder → MSE loss)
2. Sparse autoencoder: adding L1 penalty on latent activations
3. Denoising autoencoder: corrupting input with Gaussian noise, training to reconstruct clean input
4. Anomaly detection: using reconstruction error as an anomaly score

## Part 1 — Forward Pass Trace

We reproduce the worked example from Note 29 exactly: a 3→2→3 autoencoder with tanh encoder and linear decoder.

In [1]:
import torch
import torch.nn as nn

# ── Encoder weights and bias (from Note 29) ───────────────────────────────────
We = torch.tensor([[ 0.5, -0.3,  0.8],
                   [-0.2,  0.6,  0.4]], dtype=torch.float64)
be = torch.tensor([0.1, -0.1], dtype=torch.float64)

# ── Decoder weights and bias ──────────────────────────────────────────────────
Wd = torch.tensor([[ 0.7, -0.4],
                   [ 0.3,  0.9],
                   [-0.5,  0.2]], dtype=torch.float64)
bd = torch.tensor([0.0, 0.1, -0.1], dtype=torch.float64)

x = torch.tensor([1.0, 0.5, -1.0], dtype=torch.float64)

# ── Encoder forward pass ──────────────────────────────────────────────────────
z_pre = We @ x + be
print('Encoder pre-activations z_pre:', z_pre.numpy().round(4))

z = torch.tanh(z_pre)
print('Latent code z (tanh):         ', z.numpy().round(4))

# ── Decoder forward pass ──────────────────────────────────────────────────────
x_hat = Wd @ z + bd
print('Reconstruction x_hat:         ', x_hat.numpy().round(4))

# ── MSE loss ──────────────────────────────────────────────────────────────────
mse = ((x - x_hat) ** 2).mean()
print('MSE loss:                     ', round(mse.item(), 4))

Encoder pre-activations z_pre: [-0.35 -0.4 ]
Latent code z (tanh):          [-0.3364 -0.3799]
Reconstruction x_hat:          [-0.0835 -0.3429 -0.0078]
MSE loss:                      0.9563


Expected values from Note 29:
- `z_pre` = [-0.35, -0.40]
- `z` = [-0.3364, -0.3799]
- `x_hat` = [-0.0835, -0.3428, -0.0078]
- MSE = 1.2896

## Part 2 — Sparse Autoencoder

We build a small autoencoder on synthetic 8-dimensional data and add an L1 penalty on the latent activations. The total loss is:

$$\mathcal{L}_{\text{sparse}} = \text{MSE}(x, \hat{x}) + \lambda \|z\|_1$$

In [2]:
torch.manual_seed(0)

# ── Synthetic data: 200 samples, d=8 ─────────────────────────────────────────
X = torch.randn(200, 8, dtype=torch.float32)

class SparseAutoencoder(nn.Module):
    def __init__(self, d, k):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d, k), nn.Tanh())
        self.decoder = nn.Linear(k, d)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

model = SparseAutoencoder(d=8, k=16)   # overcomplete: k > d
optimiser = torch.optim.Adam(model.parameters(), lr=1e-2)
lam = 0.01

for epoch in range(300):
    x_hat, z = model(X)
    loss = ((X - x_hat) ** 2).mean() + lam * z.abs().mean()
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

x_hat_final, z_final = model(X)
recon_loss = ((X - x_hat_final) ** 2).mean().item()
mean_activation = z_final.abs().mean().item()
frac_near_zero = (z_final.abs() < 0.05).float().mean().item()

print(f'Reconstruction loss:          {recon_loss:.4f}')
print(f'Mean |z|:                     {mean_activation:.4f}')
print(f'Fraction of |z| < 0.05:       {frac_near_zero:.3f}')

Reconstruction loss:          0.0040
Mean |z|:                     0.3100
Fraction of |z| < 0.05:       0.094


The L1 penalty drives most latent units near zero for each input — the fraction near zero should be high, confirming sparsity.

## Part 3 — Denoising Autoencoder

We train a denoising autoencoder on the same synthetic data. Gaussian noise is added to the input during training; the target is always the clean input.

In [3]:
torch.manual_seed(1)

class DenoisingAutoencoder(nn.Module):
    def __init__(self, d, k):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d, k), nn.Tanh())
        self.decoder = nn.Linear(k, d)

    def forward(self, x_noisy):
        return self.decoder(self.encoder(x_noisy))

dae = DenoisingAutoencoder(d=8, k=4)
optimiser = torch.optim.Adam(dae.parameters(), lr=1e-2)
noise_std = 0.5

for epoch in range(400):
    x_noisy = X + noise_std * torch.randn_like(X)   # corrupt input
    x_hat = dae(x_noisy)                             # reconstruct
    loss = ((X - x_hat) ** 2).mean()                 # loss vs clean target
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

# Evaluate on clean input (no noise at test time)
with torch.no_grad():
    x_hat_clean = dae(X)
    clean_loss = ((X - x_hat_clean) ** 2).mean().item()

print(f'Reconstruction loss (clean input at test time): {clean_loss:.4f}')

Reconstruction loss (clean input at test time): 0.4536


## Part 4 — Anomaly Detection

We train an autoencoder on one class of data ("normal"), then measure reconstruction error on both normal and anomalous inputs. Anomalies should have higher reconstruction error.

In [4]:
torch.manual_seed(2)

# Normal data: centred near [1, 1, ..., 1]
X_normal = torch.randn(200, 8) + 1.0

# Anomalous data: centred near [-5, -5, ..., -5] — far from training distribution
X_anomaly = torch.randn(20, 8) - 5.0

class Autoencoder(nn.Module):
    def __init__(self, d, k):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d, k), nn.Tanh())
        self.decoder = nn.Linear(k, d)

    def forward(self, x):
        return self.decoder(self.encoder(x))

ae = Autoencoder(d=8, k=3)
optimiser = torch.optim.Adam(ae.parameters(), lr=1e-2)

for epoch in range(500):
    x_hat = ae(X_normal)
    loss = ((X_normal - x_hat) ** 2).mean()
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

with torch.no_grad():
    err_normal  = ((X_normal  - ae(X_normal))  ** 2).mean(dim=1)
    err_anomaly = ((X_anomaly - ae(X_anomaly)) ** 2).mean(dim=1)

print(f'Mean reconstruction error — normal:   {err_normal.mean().item():.4f}')
print(f'Mean reconstruction error — anomaly:  {err_anomaly.mean().item():.4f}')

Mean reconstruction error — normal:   0.6040
Mean reconstruction error — anomaly:  22.2127


The anomaly reconstruction error should be substantially higher than the normal reconstruction error, confirming that the autoencoder has learned the normal data distribution and cannot reconstruct out-of-distribution inputs.

## Summary

This notebook demonstrated:
1. Forward pass through a 3→2→3 autoencoder, reproducing the Note 29 worked example exactly
2. Sparse autoencoder with L1 penalty on latent activations — overcomplete bottleneck with sparsity constraint
3. Denoising autoencoder — corrupted input, clean reconstruction target, robust feature learning
4. Anomaly detection — reconstruction error as anomaly score, normal vs anomalous inputs